# Lab 10.10 &mdash; Debug & Fix the Loop

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Run a broken agent and inspect its intermediate steps
- Diagnose the wrong-tool bug from the trace
- Apply the fix (a grounding tool) and verify it improved

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; guardrail logic, tool wiring, agent structure, and reading a fixed real message trace &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Debugging, in code](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-10-10"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
The full debug loop (deck slides 14&ndash;16): run the agent with the trace visible, **read** it to
localise the bug, **diagnose** the failure mode, **fix** at the right layer, and **verify** with a re-run.
Here a buggy agent calls a tool it wasn't given (wrong tool) and then computes on an **ungrounded** number;
the fix is to give it a **grounding** tool and confirm the output is now grounded and correct.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool

from langchain_core.tools import tool
from langchain_core.messages import AIMessage, ToolMessage

@tool
def extract_figure(name: str) -> dict:
    """Ground a figure with its source from the filing."""
    return {"revenue": {"value": 120.0, "source": "p4"}}.get(name, {})
@tool
def compute(expression: str) -> float:
    """Exact arithmetic."""
    return safe_calc(expression)

# Two recorded RUN TRACES (real message lists) -- the buggy run and the fixed run:
BUGGY_TRACE = [
    AIMessage(content="", tool_calls=[{"name": "lookup_order", "args": {"q": "revenue"}, "id": "a"}]),  # wrong tool
    ToolMessage(content="unknown tool: lookup_order", tool_call_id="a"),
    AIMessage(content="", tool_calls=[{"name": "compute", "args": {"expression": "0.15*100"}, "id": "b"}]),  # 100 ungrounded
    ToolMessage(content="15.0", tool_call_id="b"),
    AIMessage(content="~15M (ungrounded)"),
]
FIXED_TRACE = [
    AIMessage(content="", tool_calls=[{"name": "extract_figure", "args": {"name": "revenue"}, "id": "a"}]),
    ToolMessage(content="{'value': 120.0, 'source': 'p4'}", tool_call_id="a"),
    AIMessage(content="", tool_calls=[{"name": "compute", "args": {"expression": "0.15*120"}, "id": "b"}]),
    ToolMessage(content="18.0", tool_call_id="b"),
    AIMessage(content="18.0M [p4]"),
]
GROUNDED = {"120"}   # the only figure actually retrieved from the filing (extract_figure -> value 120)
print("buggy & fixed traces ready")

## Your Turn
Complete `diagnose` (read the trace for the failure mode), `ungrounded_compute` (catch the ungrounded
number the same way you did in Lab 10.3), `final_of` (the answer), and give the fixed agent its
**grounding** tool. The traces above are real message lists -- the fixes are things you can grade without
running an LLM.

> **Bridge from Lab 10.3:** there you read a trace as `(tool, arg, observation)` tuples; here it is the
> **same idea with real objects** &mdash; an `AIMessage` carries the `tool_calls` (the tool name + its
> `args`) and a `ToolMessage` carries the observation. Reading the trace is identical; only the objects
> are real LangChain messages now.

In [ ]:
from langchain_core.messages import AIMessage
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

def tools_used(messages):
    return [tc['name'] for m in messages for tc in (getattr(m, 'tool_calls', None) or [])]

def diagnose(messages):
    # read the trace: what went wrong? (scan each message content for the symptom)
    for m in messages:
        if ___:   # TODO: this message content reports an unknown-tool error
            return "wrong_tool"
    return "ok"

def ungrounded_compute(messages, grounded):
    # same idea as Lab 10.3, now over REAL messages: a compute tool-call whose expression
    # uses no grounded value. An AIMessage carries tool_calls, each a {'name', 'args'} dict.
    for m in messages:
        for tc in (getattr(m, 'tool_calls', None) or []):
            if tc["name"] == "compute" and ___:   # TODO: True when NO grounded value is inside tc["args"]["expression"]
                return True
    return False

def final_of(messages):
    for m in reversed(messages):
        if isinstance(m, AIMessage) and m.content:
            return m.content
    return None

def buggy_agent():
    return create_agent(ChatOllama(model='llama3.2:1b'), [compute])   # bug: NO grounding tool

def fixed_agent():
    return create_agent(ChatOllama(model="llama3.2:1b"), ___)   # TODO: the read-only grounding + compute tools this agent was missing

try:
    print('buggy tools :', tools_used(BUGGY_TRACE))
    print('diagnosis   :', diagnose(BUGGY_TRACE))
    print('buggy final :', final_of(BUGGY_TRACE))
    print('buggy ungrounded?', ungrounded_compute(BUGGY_TRACE, GROUNDED))
    print('fixed tools :', tools_used(FIXED_TRACE))
    print('fixed final :', final_of(FIXED_TRACE))
    print('fixed ungrounded?', ungrounded_compute(FIXED_TRACE, GROUNDED))
    print('fix adds grounding tool?', 'extract_figure' not in tools_used(BUGGY_TRACE) and 'extract_figure' in tools_used(FIXED_TRACE))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the buggy trace is diagnosed as wrong_tool", lambda: diagnose(BUGGY_TRACE) == "wrong_tool")
expect_true("the buggy trace computes on an ungrounded number (100 not grounded)", lambda: ungrounded_compute(BUGGY_TRACE, GROUNDED) is True)
expect_true("the fixed trace computes only on grounded values", lambda: ungrounded_compute(FIXED_TRACE, GROUNDED) is False)
expect_true("the buggy final answer is ungrounded (no citation)", lambda: "[p" not in final_of(BUGGY_TRACE))
expect_true("the fixed trace grounds first via extract_figure", lambda: tools_used(FIXED_TRACE)[0] == "extract_figure")
expect_true("the fixed final answer is grounded (cites p4)", lambda: "[p4]" in final_of(FIXED_TRACE))
expect_true("the buggy agent lacks the grounding tool", lambda: type(buggy_agent()).__name__ == "CompiledStateGraph")
expect_true("the fix binds the grounding tool", lambda: [t.name for t in [extract_figure, compute]] == ["extract_figure", "compute"] and type(fixed_agent()).__name__ == "CompiledStateGraph")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Run the FIXED agent for real and confirm it grounds via extract_figure before computing. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
try:
    if ollama_up():
        result = fixed_agent().invoke({"messages": [{"role": "user",
                 "content": "Use extract_figure to get revenue, then compute 15% of it. Cite the page."}]},
                 config={"recursion_limit": 8})
        print("fixed tools live:", tools_used(result["messages"]))
        print("fixed final     :", final_of(result["messages"]))
    else:
        print("No Ollama reachable -- skipping the live run. The recorded traces above already show the bug & the fix.")
    print("How to debug: read the trace -> diagnose -> fix at the right layer (add the grounding tool) -> verify.")
except Exception as e:
    print("Live run skipped:", type(e).__name__)

---
### Key takeaway
Run -> read the trace -> diagnose -> fix at the right layer -> verify. The buggy agent called a tool it lacked and computed on an ungrounded number; giving it a grounding tool fixed both, and the re-run proved it. That's the debug loop.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>